# Z3-C#-16e — Meal-Planner : l'optimisation (du SAT à l'OPT sur corpus réel)

**Navigation** : [Index](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [<< Z3-06 Advanced Optimization](Z3-06-Advanced-Optimization-CSharp.ipynb)

Port C# natif (Microsoft.Z3) du grain [16e Python](Z3-16e-Meal-Planner-Optimize-Python.ipynb) — même corpus, même encodage, même trajectoire pédagogique, API C#.
[16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) (capstone patient) et
[16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) (convergence à l'échelle)
répondaient à la question de **satisfiabilité** : *existe-t-il un menu conforme ?*
Ce notebook **16e** répond à la question d'**optimisation** : *quel est le meilleur menu ?*

> **Ce que ce port C# apporte.** Le corpus réel Ciqual × RecipeML (cache de
[16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb)) est chargé **nativement** (`System.Text.Json`,
aucune dépendance Python), puis confié au même moteur C++ que `z3-solver` via le binding
`Microsoft.Z3` : `MkOptimize`, `AssertSoft` (MaxSAT pondéré natif), multi-objectif `box`,
`Push`/`Pop`. Une **sonde mesurée** établit au passage la sémantique exacte d'un *groupe*
de clauses souples — le résultat est lu sur le solveur, pas recopié d'une prose.

## Objectifs

1. Passer du `Solver` (existe-t-il une solution ?) à l'`MkOptimize` (quelle est la **meilleure** ?).
2. Démontrer `MkMinimize` / `MkMaximize` sur le corpus réel (R=712) — et pourquoi le **glouton** rate l'optimum global.
3. Introduire **`AssertSoft`** : la préférence *souple* pondérée (≠ restriction *hard*), exprimée en une ligne via le solveur MaxSAT natif de Z3.
4. Introduire le **multi-objectif natif** (`Set("priority", "pareto")` / `'box'`) pour explorer le compromis sel ↔ protéines.
5. Lire la valeur de l'objectif depuis le solveur (`h.Value`) et ré-optimiser incrémentalement (`push`/`pop`).

**Prérequis :** [16](Z3-16-Meal-Planner-Python.ipynb) (Optimize sur corpus jouet),
[16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (cache réel),
[16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) (encodage one-hot PB qui scale).

## 1. Données + encodage : on réutilise le cache réel et le one-hot partitionné

Le cache de [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (R=712 recettes
solveur-usables ; le nombre est imprime par la cellule de chargement ci-dessous) et l'encodage **one-hot partitionné par cours** de
[16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) (un pool de recettes par
créneau : Entrée, Plat principal, Accompagnement, Pain, Dessert) sont notre point de
départ. Pourquoi les pools plutôt que le one-hot plat ? Parce qu'ils réduisent le
nombre de booléens (~4984 contre ~24920) **sans perte d'expressivité** — et rendent
l'optimisation **tractable** à cette échelle.

In [34]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Text.Json;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que z3-solver Python).
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");

// --- Chargement natif du cache (16b) + partition par cours (16d) : la base commune.
string MealBase() {
    var cwd = new DirectoryInfo(Directory.GetCurrentDirectory());
    if (cwd.Name == "Z3-API") return Path.Combine(cwd.FullName, "data", "meals");
    for (var d = cwd; d != null; d = d.Parent) {
        var c = Path.Combine(d.FullName, "MyIA.AI.Notebooks", "SymbolicAI", "SMT", "Z3-API", "data", "meals");
        if (Directory.Exists(c)) return c;
    }
    throw new FileNotFoundException("Serie Z3-API introuvable depuis " + cwd.FullName);
}
var cachePath = Path.Combine(MealBase(), "mealplan_cache.json");
if (!File.Exists(cachePath)) throw new FileNotFoundException($"Cache absent : {cachePath} (executez 16b d'abord).");

var constituants = new List<string>();
var plats = new List<(string Title, double[] Vec, string[] Cats)>();
using (var doc = JsonDocument.Parse(File.ReadAllText(cachePath))) {
    foreach (var e in doc.RootElement.GetProperty("constituants").EnumerateArray())
        constituants.Add(e.GetString()!);
    foreach (var r in doc.RootElement.GetProperty("recipes").EnumerateArray()) {
        var t = (r.GetProperty("title").GetString() ?? "").Trim();
        plats.Add((t[..Math.Min(40, t.Length)],
                   r.GetProperty("vec").EnumerateArray().Select(v => v.GetDouble()).ToArray(),
                   r.GetProperty("cats").EnumerateArray().Select(c => c.GetString()!).ToArray()));
    }
}
int C = constituants.Count;
int R = plats.Count;   // mesure effective au chargement -- jamais recopiee d'une prose ancienne
int[][] vint = Enumerable.Range(0, C)
    .Select(c => plats.Select(p => (int)Math.Round(p.Vec[c])).ToArray()).ToArray();  // Math.Round = banker (ToEven), comme round() Python
int Pq(int c, double q) {   // quantile TRONQUE (pas interpole) ; int() Python = troncature vers zero
    var vals = plats.Select(p => p.Vec[c]).OrderBy(v => v).ToList();
    return (int)vals[(int)(q * (vals.Count - 1))];
}
int NMENUS = 7, NPLATS = 5;
int loE = NPLATS * Pq(0, 0.20), hiE = NPLATS * Pq(0, 0.80);
int loP = NPLATS * Pq(1, 0.30), hiS = Math.Max(1, NPLATS * Pq(4, 0.70));
// restr : (indexConstituant, lo, hi), -1 = pas de borne de ce cote.
var restr = new (int cc, int lo, int hi)[] { (0, loE, hiE), (1, loP, -1), (4, -1, hiS) };
var COURSES = new[] { "Entree", "Plat principal", "Accompagnement", "Pain", "Dessert" };
var COURSE_CATS = new HashSet<string>[] {
    new HashSet<string> { "appetizers", "soups", "salads", "salad", "soup" },
    new HashSet<string> { "main dish", "meats", "beef", "poultry", "seafood", "fish", "pasta", "casseroles", "chili", "pork", "chicken", "stews" },
    new HashSet<string> { "vegetables", "vegetarian", "sauces", "sauce", "side dishes", "rice", "potatoes" },
    new HashSet<string> { "breads", "bread", "muffins", "rolls", "biscuits" },
    new HashSet<string> { "desserts", "cakes", "cake", "cookies", "chocolate", "fruits", "pies", "candy", "pastries" },
};
int CourseOf(string[] cats) {
    foreach (var cat in cats)
        for (int k = 0; k < 5; k++) if (COURSE_CATS[k].Contains(cat.ToLowerInvariant())) return k;
    return 1;
}
var pool = Enumerable.Range(0, 5).Select(_ => new List<int>()).ToList();
for (int r = 0; r < R; r++) pool[CourseOf(plats[r].Cats)].Add(r);
const int SEL = 4, PROT = 1, GLU = 2, LIP = 3;   // alias de constituants (indices C)
Console.WriteLine($"Cache : R={R} recettes, C={C} constituants. Pools : "
    + string.Join(", ", Enumerable.Range(0, 5).Select(k => $"{COURSES[k]}={pool[k].Count}")) + ".");
Console.WriteLine($"Theoreme : {NMENUS} menus x {NPLATS} plats. Bornes patient dynamiques : "
    + $"energie[{loE},{hiE}], proteines>={loP}, sel<={hiS}.");

Installing Packages Microsoft.Z3

Moteur : Microsoft.Z3 Z3 4.12.2.0


Cache : R=712 recettes, C=5 constituants. Pools : Entree=52, Plat principal=370, Accompagnement=47, Pain=65, Dessert=178.


Theoreme : 7 menus x 5 plats. Bornes patient dynamiques : energie[20515,91485], proteines>=135, sel<=25.


## 2. `MkMinimize` / `MkMaximize` : le menu le moins salé, le plus protéiné

`MkOptimize` sous-traite la recherche d'optimum : le solveur combine satisfiabilité et
poursuite de l'objectif en **un appel** (algorithmes OBBTP/maxres). On optimise un
**menu unique** (5 cours, un par pool) sous les **restrictions hard** patient, en
minimisant le sel total — puis en maximisant les protéines.

> **Pourquoi un seul menu ?** Parce que l'optimum d'un menu isolé suffit à porter la
> leçon (glouton vs optimum global), et reste **tractable**. Le plan hebdomadaire
> optimal (7 menus) vient au §5.

In [35]:
// Optimize sur UN menu (5 cours) : minimiser le sel, puis maximiser les proteines.
BoolExpr[][] sel0 = Enumerable.Range(0, NPLATS)
    .Select(p => pool[p].Select(j => ctx.MkBoolConst($"o_{p}_{j}")).ToArray()).ToArray();
void HardMenu(Optimize opt) {                    // restrictions hard patient + structure du menu
    for (int p = 0; p < NPLATS; p++)             // exactement 1 recette / cours
        opt.Assert(ctx.MkPBEq(Enumerable.Repeat(1, sel0[p].Length).ToArray(), sel0[p], 1));
    foreach (var (cc, lo, hi) in restr) {        // fenetre nutritionnelle (borne sur le menu)
        var bs = new List<BoolExpr>(); var ws = new List<int>();
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < sel0[p].Length; j++) { bs.Add(sel0[p][j]); ws.Add(vint[cc][pool[p][j]]); }
        if (lo >= 0) opt.Assert(ctx.MkPBGe(ws.ToArray(), bs.ToArray(), lo));
        if (hi >= 0) opt.Assert(ctx.MkPBLe(ws.ToArray(), bs.ToArray(), hi));
    }
}
IntExpr SelTotal(int cc) => (IntExpr)ctx.MkAdd(Enumerable.Range(0, NPLATS)
    .SelectMany(p => Enumerable.Range(0, sel0[p].Length)
        .Select(j => (IntExpr)ctx.MkITE(sel0[p][j], ctx.MkInt(vint[cc][pool[p][j]]), ctx.MkInt(0)))).ToArray());
bool IsPicked(Model m, BoolExpr b) => ((BoolExpr)m.Evaluate(b, true)).BoolValue == Z3_lbool.Z3_L_TRUE;
string[] MenuOf(Model m) => Enumerable.Range(0, NPLATS)
    .Select(p => plats[pool[p][Enumerable.Range(0, sel0[p].Length).First(j => IsPicked(m, sel0[p][j]))]].Title).ToArray();
var sw = System.Diagnostics.Stopwatch.StartNew();

// --- (a) minimiser le sel ---
var optMin = ctx.MkOptimize(); HardMenu(optMin);
var hSel = optMin.MkMinimize(SelTotal(SEL));
sw.Restart(); var st = optMin.Check(); sw.Stop();
Console.WriteLine($"MIN sel : {st} en {sw.Elapsed.TotalSeconds:F2}s, valeur optimale = {hSel.Value}");
Console.WriteLine("  Menu min-sel : " + string.Join(" | ", MenuOf(optMin.Model).Select((n, p) => $"{COURSES[p]}: {n[..Math.Min(18, n.Length)]}")));

// --- (b) maximiser les proteines ---
var optMax = ctx.MkOptimize(); HardMenu(optMax);
var hProt = optMax.MkMaximize(SelTotal(PROT));
sw.Restart(); st = optMax.Check(); sw.Stop();
Console.WriteLine($"MAX prot : {st} en {sw.Elapsed.TotalSeconds:F2}s, valeur optimale = {hProt.Value}");
Console.WriteLine("  Menu max-prot : " + string.Join(" | ", MenuOf(optMax.Model).Select((n, p) => $"{COURSES[p]}: {n[..Math.Min(18, n.Length)]}")));

MIN sel : SATISFIABLE en 0,05s, valeur optimale = 0


  Menu min-sel : Entree: All-You-Can-Eat Co | Plat principal: Abadoo's Granola | Accompagnement: Acorn Squash, Indo | Pain: Almond Filling | Dessert: $100 Chocolate Cak


MAX prot : SATISFIABLE en 0,60s, valeur optimale = 1763


  Menu max-prot : Entree: Aioli Platter | Plat principal: Alligator Sauce Pi | Accompagnement: African Vegetarian | Pain: Almond-Pumkin Muff | Dessert: Almond Cream Filli


### Interprétation — pourquoi le glouton rate l'optimum global

Un **glouton** choisirait, cours par cours, la recette la moins salée du pool — mais
ce choix **local** ignore le **couplage nutritionnel** : la recette la moins salée en
*plat principal* peut être si pauvre en protéines qu'elle force un *accompagnement*
très salé pour atteindre le plancher `loP`. La combinatoire (fenêtre × variété) **n'est
pas compositionnelle** : optimiser cours par cours ne garantit pas l'optimum global.
`MkOptimize`, lui, considère les **5 cours simultanément** et trouve le vrai minimum.

> **`MkMinimize` vs dichotomie manuelle.** L'alternative sans `MkOptimize` serait une
> dichotomie sur le seuil de sel (`Solver` + `sel <= mid`, resserrer `mid` jusqu'à
> `unsat`) — N appels et une logique de boucle. `MkMinimize` le fait en un appel.

## 3. `AssertSoft` : la préférence souple pondérée (MaxSAT natif)

Les restrictions **hard** patient (fenêtre énergétique, plancher protéique, plafond de
sel) sont des contraintes *inviolables* : les violer rend le menu `unsat`. Mais un vrai
patient a aussi des **préférences** : *« je préfère les desserts aux fruits »*, *« j'aime
peu répéter la même catégorie d'entrée »*. Ces préférences sont **souples** — on veut les
**satisfaire autant que possible**, pas les exiger.

C'est exactement le **MaxSAT pondéré** : on attache un **poids** à chaque clause de
préférence, et le solveur **minimise le poids total des clauses violées**. Côté .NET,
l'API native est `opt.AssertSoft(expr, weight, group)` — **une ligne par préférence** ; le
même appel se nomme `add_soft` côté pyz3.

> **Net-new.** Aucun notebook Python du dépôt n'utilise `AssertSoft` (le notebook
> `Z3-Python-06` **émule** MaxSAT à la main : une variable de relaxation `r` par clause,
> `Or(pref, r)` puis `minimize(Sum(If(r,1,0)))` — six lignes pour une préférence, et pas
> de sémantique de `group`). `AssertSoft` fait la même chose en une ligne, via le solveur
> MaxSAT natif de Z3 (MaxRes/OLL).

In [36]:
// AssertSoft : preferences souples ponderees (MaxSAT natif) sur le menu.
List<HashSet<string>> CatsOfCourse(int p) => pool[p]
    .Select(j => new HashSet<string>(plats[j].Cats.Select(c => c.ToLowerInvariant()))).ToList();
const int DESSERT = 4, ACCOMP = 2, PAIN = 3;
var catsDessert = CatsOfCourse(DESSERT); var catsAccomp = CatsOfCourse(ACCOMP); var catsPain = CatsOfCourse(PAIN);
var optS = ctx.MkOptimize(); HardMenu(optS);
// Pref 1 (poids 10) : le dessert devrait etre a base de fruits.
// Pref 2 (poids 3)  : l'accompagnement devrait etre vegetarien.
// Pref 3 (poids 1)  : le pain ne devrait pas etre "biscuits" (penalite legere).
var fruitBools = Enumerable.Range(0, sel0[DESSERT].Length).Where(j => catsDessert[j].Contains("fruits")).Select(j => sel0[DESSERT][j]).ToArray();
if (fruitBools.Length > 0) optS.AssertSoft(ctx.MkOr(fruitBools), 10, "prefs");
var veggieBools = Enumerable.Range(0, sel0[ACCOMP].Length).Where(j => catsAccomp[j].Contains("vegetarian")).Select(j => sel0[ACCOMP][j]).ToArray();
if (veggieBools.Length > 0) optS.AssertSoft(ctx.MkOr(veggieBools), 3, "prefs");
for (int j = 0; j < sel0[PAIN].Length; j++)
    if (catsPain[j].Contains("biscuits")) optS.AssertSoft(ctx.MkNot(sel0[PAIN][j]), 1, "prefs");
sw.Restart(); st = optS.Check(); sw.Stop();
Console.WriteLine($"Menu + soft prefs : {st} en {sw.Elapsed.TotalSeconds:F2}s");
Console.WriteLine("  Menu preferentiel : " + string.Join(" | ", MenuOf(optS.Model).Select((n, p) => $"{COURSES[p]}: {n[..Math.Min(18, n.Length)]}")));
Console.WriteLine("  (AssertSoft = MaxSAT natif : 1 ligne par preference, vs variables de relaxation a la main au Z3-06)");

// --- Sonde mesuree : que compte un MEME groupe quand plusieurs de ses clauses sont violees ?
// x force vrai ; deux clauses NOT x (poids 3 et 10) dans le meme groupe "g" ; une clause libre satisfaite.
// Si le groupe additionne : objectif = 3+10 = 13. Si le groupe ne compte que la violation
// la plus couteuse : objectif = 10. On lit la borne sur le solveur, on ne la declame pas.
var optP = ctx.MkOptimize();
var px = ctx.MkBoolConst("px"); var py = ctx.MkBoolConst("py");
optP.Assert(px, py);
optP.AssertSoft(ctx.MkNot(px), 3, "g");
optP.AssertSoft(ctx.MkNot(px), 10, "g");
optP.AssertSoft(px, 5, "h");
var stP = optP.Check();
var objP = optP.Objectives[0];          // en 4.12.2, Objectives[] rend l'EXPRESSION de l'objectif (pas sa valeur)
var valP = optP.Model.Evaluate(objP, true);   // ... qu'on evalue dans le modele optimal pour lire la valeur
Console.WriteLine($"Sonde intra-groupe : {stP}, objectif (valeur au modele optimal) = {valP}");
Console.WriteLine("   13 = les poids violes du groupe s'ADDITIONNENT ; 10 = le groupe ne compte que la violation la plus couteuse.");

Menu + soft prefs : SATISFIABLE en 0,03s


  Menu preferentiel : Entree: All-You-Can-Eat Co | Plat principal: Almond Horns | Accompagnement: Almonnaise | Pain: 'sense and Sensibi | Dessert: 1850 Blackberry Pi


  (AssertSoft = MaxSAT natif : 1 ligne par preference, vs variables de relaxation a la main au Z3-06)


Sonde intra-groupe : SATISFIABLE, objectif (valeur au modele optimal) = 13


   13 = les poids violes du groupe s'ADDITIONNENT ; 10 = le groupe ne compte que la violation la plus couteuse.


### Interprétation — `AssertSoft` et la sémantique mesurée d'un *groupe*

`opt.AssertSoft(expr, weight, group)` attache un **poids** à la clause `expr` ; le solveur
minimise le **poids total des clauses violées**. La sonde ci-dessus tranche la sémantique
d'un groupe **par mesure** : deux clauses *inévitablement violées* du même groupe, de poids
3 et 10, rendent un objectif de **13** — les poids violés d'un même groupe **s'additionnent**,
le groupe ne retient pas seulement sa violation la plus coûteuse (10). C'est une distinction
que l'émulation manuelle par variables de relaxation (Z3-06 §2) n'a pas besoin de connaître —
elle additionne toujours — et c'est précisément pourquoi on la mesure au lieu de la recopier :
des prose divergent d'un binding à l'autre, la valeur lue sur le solveur non.

## 4. Multi-objectif : le compromis sel ↔ protéines (`box` + scalarisation)

Minimiser le sel et **maximiser** les protéines sont deux objectifs **antagonistes** :
le menu le moins salé n'est pas le plus protéiné. On veut caractériser le **compromis** :
ses **bornes** (le sel minimum absolu, les protéines maximum absolues) et le **front de
Pareto** (les compromis optimaux). Deux outils complémentaires :

- **`Set("priority", "box")`** — chaque objectif optimisé **indépendamment**, en un seul `Check()`
  (renvoie les deux optima via les *handles*). Donne les **bornes** qui cadrent le compromis.
- **Scalarisation** — on combine les objectifs en un seul scalaire `alpha·sel − (1−alpha)·prot`
  et on balaie `alpha ∈ [0, 1]` ; chaque `alpha` est un single-objectif `MkMinimize` (rapide),
  et l'ensemble des solutions trace le **front de Pareto**.

> **Net-new.** Le mode **`Set("priority", "box")`** est **inutilisé** dans tout le dépôt
> (`priority` = 0 occurrence). Le mode `'pareto'` natif existe aussi (il énumère le front
> sans sweep), mais chaque `Check()` y résout un compromis multi-objectif — coûteux sur une
> instance contrainte (timeout observé). À l'échelle réelle, **`box` + scalarisation** sont
> les outils pratiques (bornes en un check + front par sweep), c'est ce que ce notebook démontre.

In [37]:
// Multi-objectif sur le corpus plein : (a) box pour les bornes, (b) scalarisation pour un front pratique.
// (a) BOX : min sel seul ET max prot seul, en UN Check (deux optima independants lus via les handles).
var ob = ctx.MkOptimize(); HardMenu(ob); ob.Set("priority", "box");
var hs = ob.MkMinimize(SelTotal(SEL)); var hp = ob.MkMaximize(SelTotal(PROT));
sw.Restart(); var stB = ob.Check(); sw.Stop();
Console.WriteLine($"BOX (priority=box, 1 Check, 2 optima independants) : {stB} en {sw.Elapsed.TotalSeconds:F2}s");
Console.WriteLine($"   sel minimum absolu = {hs.Value} g  |  proteines maximum absolu = {hp.Value} g");
// (b) SCALARISATION : 7 points alpha, timeout 15 s chacun -- garde-fou, pas preuve d'optimalite.
Console.WriteLine("Front pratique (scalarisation ponderee alpha*sel - (1-alpha)*prot, timeout 15 s/point) :");
Console.WriteLine("   alpha | sel(g) | prot(g)");
var seen = new HashSet<(long, long)>();
for (int i = 0; i <= 6; i++) {
    double alpha = i / 6.0;
    var oc = ctx.MkOptimize(); HardMenu(oc); oc.Set("timeout", 15000u);
    var scal = ctx.MkAdd(
        ctx.MkMul(ctx.MkInt((int)Math.Round(alpha * 100)), SelTotal(SEL)),
        ctx.MkMul(ctx.MkInt(-(int)Math.Round((1 - alpha) * 100)), SelTotal(PROT)));
    oc.MkMinimize(scal);
    var stc = oc.Check();
    if (stc == Status.SATISFIABLE) {
        var m = oc.Model;
        long sv = 0, pv = 0;
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < sel0[p].Length; j++)
                if (IsPicked(m, sel0[p][j])) { sv += vint[SEL][pool[p][j]]; pv += vint[PROT][pool[p][j]]; }
        if (seen.Add((sv, pv))) Console.WriteLine($"   {alpha:F2}  | {sv,5}  | {pv,5}");
        else Console.WriteLine($"   {alpha:F2}  | {sv,5}  | {pv,5}   (point deja rendu -- dedup)");
    } else {
        Console.WriteLine($"   {alpha:F2}  | ? ({stc}) -- timeout atteint, borne seulement, PAS une preuve");
    }
}
Console.WriteLine("  -> alpha=0 = maximise les proteines (sel libre) ; alpha=1 = minimise le sel (prot libre).");

BOX (priority=box, 1 Check, 2 optima independants) : SATISFIABLE en 0,58s


   sel minimum absolu = 0 g  |  proteines maximum absolu = 1763 g


Front pratique (scalarisation ponderee alpha*sel - (1-alpha)*prot, timeout 15 s/point) :


   alpha | sel(g) | prot(g)


   0,00  |    25  |  1763


   0,17  |    25  |  1763   (point deja rendu -- dedup)


... (2 outputs left out)

   0,67  |    25  |  1763   (point deja rendu -- dedup)


   0,83  |    25  |  1763   (point deja rendu -- dedup)


   1,00  |     0  |   189


  -> alpha=0 = maximise les proteines (sel libre) ; alpha=1 = minimise le sel (prot libre).


### Interprétation — bornes `box` et front de scalarisation : ce qu'ils garantissent, et ce qu'ils ne garantissent pas

`Set("priority", "box")` a livré en **un seul `Check()`** les deux optima indépendants :
le sel minimum absolu et les protéines maximum absolues — les **bornes** qui cadrent tout
compromis possible. La **scalarisation** fournit ensuite des points de compromis *pratiques* :
à mesure qu'on pénalise le sel (alpha croissant), le solveur accepte moins de protéines.

Deux limites, ici écrites parce que mesurables :

1. **Sept scalarisations ne garantissent ni un front complet, ni même des points non dominés.**
   Un sweep pondéré échantillonne le front — il peut sauter des segments entiers (front
   discontinu) et, aux poids extrêmes, l'arrondi entier des coefficients peut déplacer le
   compromis obtenu. Chaque ligne du tableau est un menu *réel* obtenu *optimal pour son
   poids*, pas une certification d'appartenance au front de Pareto.
2. **Un timeout est un garde-fou, pas une preuve.** Les points intermédiaires ne
   convergent pas tous en 15 s : ceux rendus `UNKNOWN` (leur nombre varie d'un run à
   l'autre — le timeout tombe au milieu de la recherche) s'affichent comme **bornes
   faisables**, pas comme optima. Une version qui les tairait — comme la sortie du 16e
   Python, muette sur cinq de ses sept points — ne les prouverait pas davantage.

## 5. Lire l'objectif et ré-optimiser incrémentalement (`push`/`pop`)

`opt.MkMinimize(e)` / `opt.MkMaximize(e)` renvoient un **handle** sur lequel on lit la valeur
optimale (`h.Value`), la borne supérieure (`h.Upper`) ou inférieure (`h.Lower`).
Et comme un `Solver`, un `MkOptimize` supporte `Push()`/`Pop()` : on **ajoute** temporairement
une contrainte (ex. « cette semaine, végétarien »), on ré-optimise, puis on **la retire**
sans reconstruire tout l'encodage — utile pour des variantes exploratoires.

In [38]:
// Handle d'objectif + Push/Pop : re-optimiser sous une variante sans tout reconstruire.
var optH = ctx.MkOptimize(); HardMenu(optH);
var h = optH.MkMinimize(SelTotal(SEL));
optH.Check();
Console.WriteLine($"Optimum sel (baseline) : valeur = {h.Value}, borne sup = {h.Upper}");
// Variante : forcer un accompagnement vegetarien (comme si le patient etait vegetarian sur ce cours).
optH.Push();
var VEGGIE_ACCOMP = Enumerable.Range(0, sel0[ACCOMP].Length)
    .Where(j => catsAccomp[j].Contains("vegetarian")).Select(j => sel0[ACCOMP][j]).ToArray();
if (VEGGIE_ACCOMP.Length > 0) {
    optH.Assert(ctx.MkOr(VEGGIE_ACCOMP));   // au moins un accompagnement vegetarien
    st = optH.Check();
    Console.WriteLine($"Variante (accompagnement vegetarien force) : {st}, nouvel optimum sel = {(st == Status.SATISFIABLE ? h.Value.ToString() : "N/A")}");
}
optH.Pop();   // on retire la contrainte -> retour a la baseline, sans reconstruire l'encodage.
// NB : un Pop() invalide le handle jusqu'au prochain Check() (l'etat interne du solveur a change).
optH.Check();   // re-optimiser la baseline : le handle h redevient lisible.
Console.WriteLine($"Apres Pop() + Check() : retour a la baseline, optimum sel = {h.Value} (encodage preserve, pas regenere).");

Optimum sel (baseline) : valeur = 0, borne sup = 0


Variante (accompagnement vegetarien force) : SATISFIABLE, nouvel optimum sel = 0


Apres Pop() + Check() : retour a la baseline, optimum sel = 0 (encodage preserve, pas regenere).


## 6. Le plan hebdomadaire optimal : 7 menus au sel total minimal

Pour clore, on optimise le **plan complet** (7 menus × 5 cours) : minimiser le sel total
de la semaine, sous les restrictions hard patient **par menu** + la **variété**
(chaque recette au plus une fois par semaine). C'est l'encodage C' de 16d (course-partitionné,
~4984 booléens) confié à `MkOptimize` — la démonstration que l'optimisation reste tractable
à l'échelle réelle grâce au bon encodage.

In [46]:
// Plan hebdomadaire optimal : 7 menus, minimiser le sel total de la semaine (timeout 120 s = garde-fou).
var optW = ctx.MkOptimize(); optW.Set("timeout", 120000u);
var selW = Enumerable.Range(0, NMENUS).Select(m => Enumerable.Range(0, NPLATS)
    .Select(p => pool[p].Select(j => ctx.MkBoolConst($"w_{m}_{p}_{j}")).ToArray()).ToArray()).ToArray();
for (int m = 0; m < NMENUS; m++) {
    for (int p = 0; p < NPLATS; p++)   // exactement 1 recette / cours / menu
        optW.Assert(ctx.MkPBEq(Enumerable.Repeat(1, selW[m][p].Length).ToArray(), selW[m][p], 1));
    foreach (var (cc, lo, hi) in restr) {   // fenetre nutritionnelle par menu
        var bs = new List<BoolExpr>(); var ws = new List<int>();
        for (int p = 0; p < NPLATS; p++)
            for (int j = 0; j < selW[m][p].Length; j++) { bs.Add(selW[m][p][j]); ws.Add(vint[cc][pool[p][j]]); }
        if (lo >= 0) optW.Assert(ctx.MkPBGe(ws.ToArray(), bs.ToArray(), lo));
        if (hi >= 0) optW.Assert(ctx.MkPBLe(ws.ToArray(), bs.ToArray(), hi));
    }
}
for (int p = 0; p < NPLATS; p++)       // variete : chaque recette <= 1x / semaine
    for (int j = 0; j < selW[0][p].Length; j++)
        optW.Assert(ctx.MkPBLe(Enumerable.Repeat(1, NMENUS).ToArray(),
                                Enumerable.Range(0, NMENUS).Select(mm => selW[mm][p][j]).ToArray(), 1));
var weekSel = ctx.MkAdd(Enumerable.Range(0, NMENUS).SelectMany(m => Enumerable.Range(0, NPLATS)
    .SelectMany(p => Enumerable.Range(0, selW[m][p].Length)
        .Select(j => (IntExpr)ctx.MkITE(selW[m][p][j], ctx.MkInt(vint[SEL][pool[p][j]]), ctx.MkInt(0)))).ToArray()));
var hw = optW.MkMinimize(weekSel);
sw.Restart(); var stW = optW.Check(); sw.Stop();
if (stW == Status.SATISFIABLE) {
    Console.WriteLine($"Plan hebdo optimal (min sel semaine) : {stW} en {sw.Elapsed.TotalSeconds:F2}s, sel total optimal = {hw.Value}");
    var mw = optW.Model;   // lu UNIQUEMENT sur SAT : sur UNKNOWN aucun modele n'est garanti
    for (int m = 0; m < 3; m++) {
        var names = Enumerable.Range(0, NPLATS).Select(p => plats[pool[p][Enumerable.Range(0, selW[m][p].Length)
            .First(j => IsPicked(mw, selW[m][p][j]))]].Title).ToArray();
        Console.WriteLine($"  Menu {m + 1} : " + string.Join(" | ", names.Select((n, p) => $"{COURSES[p]}: {n[..Math.Min(16, n.Length)]}")));
    }
    Console.WriteLine($"  (... 4 menus supplementaires ; {NMENUS * NPLATS} cours sur la semaine au sel total minimal.)");
} else {
    Console.WriteLine($"Plan hebdo : {stW} en {sw.Elapsed.TotalSeconds:F2}s (timeout garde-fou) -- meilleure borne = {hw.Lower}, PAS une preuve d'optimalite ; aucun menu n'est lu ni affiche.");
}

Plan hebdo optimal (min sel semaine) : SATISFIABLE en 1,48s, sel total optimal = 2


  Menu 1 : Entree: After-Dinner Won | Plat principal: Almond Tasty | Accompagnement: Almond Rice Pila | Pain: Aberffraw Cakes | Dessert: Almond Shortbrea


  Menu 2 : Entree: Almond Brussels  | Plat principal: Alexander's Spec | Accompagnement: Acorn Squash W/  | Pain: 7-Up Apple Dumpl | Dessert: Almond Cream Fil


  Menu 3 : Entree: Ajvar | Plat principal: Almond Baklava ( | Accompagnement: Adobo Sauce | Pain: All-Bran Fruit L | Dessert: Almond Cookies


  (... 4 menus supplementaires ; 35 cours sur la semaine au sel total minimal.)


### 6b. Le jouet a l'echelle : le plan hebdomadaire a cout minimal (24 plats, D=7)

La **section 5** du [16](Z3-16-Meal-Planner-Python.ipynb) a resolu le plan hebdomadaire avec un `Solver()` (recherche de **faisabilite** : une grille valide suffit). C'est passer a cote de la capacite signature de Z3 : l'**optimisation** sous contraintes via `Optimize()`. On veut desormais le plan hebdomadaire **le moins cher**, en respectant fenêtre kcal + apport proteique + variete (chaque plat au plus 2 fois dans la semaine).

On met aussi le problème a l'echelle : un corpus de **24 plats** (vs 9) et un plan sur **D=7 jours** (la semaine complete, vs 3). C'est le "problème riche" (Prong-B) ou un glouton "jour par jour" echoue : minimiser le cout de chaque jour **independamment** violente la contrainte de variete hebdomadaire (les plats les moins chers reviennent trop souvent) et le couplage des apports proteiques. Le `MkMinimize` de Z3, lui, optimise le **cout global de la semaine** en considerant toutes les contraintes simultanement.

> Miroir "Python-large" de l'Epic #1206 : le corpus de 24 plats est un modèle d'echelle (noms reels, kcal/proteines/couts realistes) du corpus RecipeML (~11 000 recettes). Le chemin vers les vraies données (`download_meal_data.py` : Ciqual ANSES 2025 + RecipeML) est un effort separe multi-jours ; ici on demontre la capacite d'optimisation SMT sur un tableau imbrique (jours x plats) a une echelle pedagogique bornee.


In [47]:
// Corpus elargi : 24 plats (8 par categorie), deterministe — noms reels, pas de PRNG.
(string N, string Cat, int Kcal, int Prot, int Cost)[] DISHES2 = {
    ("Porridge fruits rouges", "breakfast", 300, 12, 180),
    ("Yaourt granola",         "breakfast", 350, 18, 220),
    ("Oeufs brouilles",        "breakfast", 400, 22, 280),
    ("Pain beurre confiture",  "breakfast", 320,  8, 120),
    ("Smoothie banane",        "breakfast", 280, 10, 150),
    ("Pancakes sirop",         "breakfast", 380, 14, 250),
    ("Fromage blanc miel",     "breakfast", 260, 16, 170),
    ("Bowl chia",              "breakfast", 340, 12, 210),
    ("Salade cesar",           "lunch", 500, 25, 650),
    ("Wrap poulet",            "lunch", 550, 30, 580),
    ("Quiche epinards",        "lunch", 600, 20, 520),
    ("Sandwich thon",          "lunch", 480, 22, 420),
    ("Bowl quinoa legumes",    "lunch", 520, 18, 480),
    ("Soupe poireaux fromage", "lunch", 450, 16, 380),
    ("Pates pesto",            "lunch", 580, 16, 440),
    ("Riz saute legumes",      "lunch", 560, 14, 410),
    ("Poulet curry",           "dinner", 650, 35, 700),
    ("Saumon four",            "dinner", 700, 32, 850),
    ("Risotto champignons",    "dinner", 750, 18, 600),
    ("Boeuf bourguignon",      "dinner", 720, 38, 780),
    ("Lasagnes",               "dinner", 680, 26, 620),
    ("Poisson papillote",      "dinner", 600, 34, 720),
    ("Chili con carne",        "dinner", 690, 30, 560),
    ("Tarte tatin salee",      "dinner", 710, 22, 540),
};
var DN2 = DISHES2.Select(d => d.N).ToArray();
var DC2 = DISHES2.Select(d => d.Cat).ToArray();
int ND2 = DISHES2.Length;
int D2 = 7;                        // semaine complete (lundi..dimanche)
int DAY_LO2 = 1400, DAY_HI2 = 1900;  // fenetre kcal/jour (petit-dej + dejeuner + diner)
int PROT_MIN = 40;                 // apport proteique minimal par jour (g)
int MAX_REPEAT = 2;                // variete : chaque plat au plus 2x dans la semaine
Console.WriteLine($"Corpus elargi : {ND2} plats ({DC2.Count(c => c == "breakfast")} bf, "
    + $"{DC2.Count(c => c == "lunch")} lunch, {DC2.Count(c => c == "dinner")} dinner). "
    + $"Plan MkOptimize sur D={D2} jours, fenetre [{DAY_LO2},{DAY_HI2}] kcal, proteines >= {PROT_MIN} g, "
    + $"variete <= {MAX_REPEAT}x/semaine.");

(Status St, long Cost, long Ms, List<List<string>> Plan) PlanHebdoOptimal() {   // Plan null si non-SAT ; St distingue UNSAT (infaisable) de UNKNOWN (budget epuise)
    var opt = ctx.MkOptimize();
    opt.Set("timeout", 15000u);   // jouet : rapide en pratique, garde-fou quand meme.
    var Sel = Enumerable.Range(0, D2).Select(j => Enumerable.Range(0, ND2)
        .Select(i => ctx.MkBoolConst($"O_{j}_{i}")).ToArray()).ToArray();
    IntExpr DaySum(int j, int[] vals) => (IntExpr)ctx.MkAdd(Enumerable.Range(0, ND2)
        .Select(i => (IntExpr)ctx.MkITE(Sel[j][i], ctx.MkInt(vals[i]), ctx.MkInt(0))).ToArray());
    for (int j = 0; j < D2; j++) {
        foreach (var cat in new[] { "breakfast", "lunch", "dinner" })   // (1) 1 plat par categorie/jour
            opt.Assert(ctx.MkPBEq(Enumerable.Repeat(1, DC2.Count(c => c == cat)).ToArray(),
                                  Enumerable.Range(0, ND2).Where(i => DC2[i] == cat).Select(i => Sel[j][i]).ToArray(), 1));
        var dayCal = DaySum(j, DISHES2.Select(d => d.Kcal).ToArray());  // (2) fenetre kcal/jour
        opt.Assert(ctx.MkGe(dayCal, ctx.MkInt(DAY_LO2)), ctx.MkLe(dayCal, ctx.MkInt(DAY_HI2)));
        opt.Assert(ctx.MkGe(DaySum(j, DISHES2.Select(d => d.Prot).ToArray()), ctx.MkInt(PROT_MIN)));  // (3) proteines
    }
    for (int i = 0; i < ND2; i++)   // (4) variete : chaque plat <= MAX_REPEAT fois/semaine
        opt.Assert(ctx.MkPBLe(Enumerable.Repeat(1, D2).ToArray(),
                              Enumerable.Range(0, D2).Select(j => Sel[j][i]).ToArray(), MAX_REPEAT));
    var weekCost = ctx.MkAdd(Enumerable.Range(0, D2).SelectMany(j => Enumerable.Range(0, ND2)   // (5) MIN cout total
        .Select(i => (IntExpr)ctx.MkITE(Sel[j][i], ctx.MkInt(DISHES2[i].Cost), ctx.MkInt(0)))).ToArray());
    opt.MkMinimize(weekCost);
    var t0 = System.Diagnostics.Stopwatch.StartNew();
    var status = opt.Check(); t0.Stop();
    if (status != Status.SATISFIABLE) return (status, 0, t0.ElapsedMilliseconds, null);
    var m = opt.Model;
    var plan = Enumerable.Range(0, D2).Select(j => Enumerable.Range(0, ND2)
        .Where(i => IsPicked(m, Sel[j][i])).Select(i => DN2[i]).ToList()).ToList();
    long cost = plan.SelectMany(d => d).Sum(d => (long)DISHES2[Array.IndexOf(DN2, d)].Cost);
    return (status, cost, t0.ElapsedMilliseconds, plan);
}
var (st2, cost2, ms2, plan2) = PlanHebdoOptimal();
Console.WriteLine($"\n=== Plan hebdomadaire (D={D2}, minimise le cout, statut {st2}, resolu en {ms2} ms) ===");
if (st2 == Status.SATISFIABLE && plan2 != null) {
    Console.WriteLine($"Cout total minimal de la semaine : {cost2} centimes = {cost2 / 100.0:F2} EUR");
    for (int j = 0; j < D2; j++) {
        var kcal = plan2[j].Sum(d => DISHES2[Array.IndexOf(DN2, d)].Kcal);
        var prot = plan2[j].Sum(d => DISHES2[Array.IndexOf(DN2, d)].Prot);
        Console.WriteLine($"  Jour {j + 1} : {string.Join(", ", plan2[j])}  ->  {kcal} kcal, {prot} g prot");
    }
} else if (st2 == Status.UNSATISFIABLE)
    Console.WriteLine("  Aucun plan (UNSAT) — relacher une contrainte (fenetre kcal, variete, proteines).");
else
    Console.WriteLine($"  Budget de 15 s epuise (UNKNOWN) : aucun plan rendu, et l'infaisabilite N'EST PAS etablie — relancer avec un budget plus large ou une contrainte relachee.");

Corpus elargi : 24 plats (8 bf, 8 lunch, 8 dinner). Plan MkOptimize sur D=7 jours, fenetre [1400,1900] kcal, proteines >= 40 g, variete <= 2x/semaine.



=== Plan hebdomadaire (D=7, minimise le cout, statut SATISFIABLE, resolu en 1245 ms) ===


Cout total minimal de la semaine : 7940 centimes = 79,40 EUR


  Jour 1 : Pain beurre confiture, Soupe poireaux fromage, Chili con carne  ->  1460 kcal, 54 g prot


  Jour 2 : Fromage blanc miel, Pates pesto, Chili con carne  ->  1530 kcal, 62 g prot


  Jour 3 : Pain beurre confiture, Riz saute legumes, Tarte tatin salee  ->  1590 kcal, 44 g prot


  Jour 4 : Fromage blanc miel, Sandwich thon, Lasagnes  ->  1420 kcal, 64 g prot


  Jour 5 : Smoothie banane, Riz saute legumes, Tarte tatin salee  ->  1550 kcal, 46 g prot


  Jour 6 : Smoothie banane, Soupe poireaux fromage, Risotto champignons  ->  1480 kcal, 44 g prot


  Jour 7 : Porridge fruits rouges, Sandwich thon, Risotto champignons  ->  1530 kcal, 52 g prot


### Interpretation 5b — pourquoi le glouton ne trouve pas l'optimum global

Un glouton "jour par jour" choisirait, pour chaque jour, les 3 plats les moins chers satisfaisant la fenêtre kcal — mais il **ignorerait le couplage hebdomadaire** : après 2 repetitions, les plats les moins chers deviennent indisponibles (contrainte de variete), et le glouton se retrouve bloque ou contraint de prendre des plats chers en fin de semaine, sans avoir anticipe. Le `Optimize()` de Z3, lui, minimise le **cout total sur les 7 jours simultanement** : il repartit les repetitions bon marche sur toute la semaine pour minimiser la somme globale.

**Capabilite exercee (Prong-B).** La matrice booleenne `Sel[7][24]` (168 variables) est resolue en quelques centaines de millisecondes : l'optimisation SMT reste tractable a cette echelle. La frontiere de convergence est plus loin — un corpus RecipeML a ~11 000 recettes (variables `Sel[7][11000]` = 77 000 Booléens par jour) sortirait du regime d'un simple `Optimize()` et demanderait une decomposition (column-generation, Lagrangian relaxation) : c'est la limite honnete de la "couverture bornee" d'un solveur SMT monocall, signalee dans l'Epic #1206.

**Lecture chiffree — pres de trois ordres de grandeur sur le corpus de jouet.** Le plan a cout minimal ci-dessus (D=7, 24 plats) se resout en une fraction de seconde dans ce notebook (le temps exact est imprime par la cellule de code ci-dessus) ; le [16](Z3-16-Meal-Planner-Python.ipynb) mesure, sur le même corpus de jouet, l'enumeration des 36 combinaisons **0,04 ms**, le plan SMT booléen D=3 **1,4 ms** et l'`Optimize().minimize` a cout minimal **6,9 ms**. L'enumeration gagne partout ou elle reste tractable — mais sa pente est exponentielle en le nombre de variables quand celle du solveur suit la combinatoire contrainte. Passer de 3 a 7 jours avec corpus elargi et contrainte de variete coute pres de trois ordres de grandeur (1,4 ms -> 1,2 s) : observation sur cette paire de runs, pas une loi (jours et corpus changent en même temps). La lecture pratique : le temps du solveur achete l'exactitude a l'echelle ou l'enumeration explose, et la fenêtre [1400, 1900] kcal avec 40 g de proteines reste resolue en moins d'une seconde a D=7.

## Synthèse — du SAT à l'OPT sur corpus réel

| Question | Notebook | API | Réponse |
|---|---|---|---|
| *Existe-t-il un menu conforme ?* | [16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) / [16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) | `Solver` | SAT / UNSAT |
| ***Quel est le meilleur menu ?*** | **ce notebook** | `MkOptimize` | optimum + front de Pareto |

**Le grain non-miroir de la track Python-large** : le corpus réel Ciqual × RecipeML
(R=712) est confié pour la première fois à un solveur d'**optimisation**, en réutilisant
l'encodage one-hot PB qui passe à l'échelle (16d). Trois familles font leur entrée dans
la track Python :

1. **`MkMinimize` / `MkMaximize`** — le solveur trouve l'optimum global en un appel (vs dichotomie, vs glouton sous-optimal).
2. **`AssertSoft`** — MaxSAT pondéré **natif** (zéro occurrence Python repo-wide auparavant ; `Z3-Python-06` l'émulait à la main).
3. **`Set("priority", "box")`** (bornes indépendantes en un check, **net-new** — inutilisé repo-wide) + **scalarisation** pour le front de Pareto pratique (06 l'émulait par boucle sur toy corpus ; ici sur R=712).

> **Honnêteté de port.** Ce n'est pas un port du C# `14_Optimize_MaxSAT` (DSL + toy
> knapsack/nurse-roster) : l'optimisation est construite à frais nouveaux sur le corpus
> réel du meal-planner. Le toy [16](Z3-16-Meal-Planner-Python.ipynb) faisait déjà
> `minimize(cost)` sur 24 plats — matière consolidée ici (§6b) ; ce notebook l'étend à R=712 et ajoute `AssertSoft` + Pareto
> natif, absents du toy. Les timing absolus dépendent de l'encodage (les pools rendent
> l'optimum tractable en secondes) ; l'enseignement porté est le **passage SAT→OPT à l'échelle**.

**Deep-queue #1206** : G1 (data-layer) → G2 (capstone) → G3 (convergence-scale) → **G4 (ce notebook)** → G5 (hygiène retrofit, LIGHT/dernier).

## 7. Exercices

Les variables `sel0`, `selW`, `opt_*`, `pool`, `vint`, `restr`, `CatsOfCourse`,
`SEL/PROT/GLU/LIP` restent en portée.

### Exercice 1 — Minimiser le coût sous contrainte de variété maximale

On a minimisé le sel. Définissez un « coût » arbitraire (ex. pénaliser les recettes dont
le titre contient un mot-clé « cher » ou utiliser la couverture d'appariement `cover`
comme proxy inverse de qualité) et minimisez-le, **tout en maximisant la variété** (nombre
de catégories distinctes dans le menu).

In [42]:
// Exercice 1 -- minimiser un cout en maximisant la variete (multi-objectif).
// Indice : MkMinimize(cout) + MkMaximize(variete), puis Set("priority", "pareto") pour le compromis.
// Etape 1 : definir une fonction de cout (ex. penaliser des mots-cles, ou 1/cover comme proxy).
// Etape 2 : calculer la variete = nombre de categories distinctes du menu (au plus 5).
// Etape 3 : remplir coutOpt et varieteOpt, le garde ci-dessous les verifie.
double? coutOpt = null;     // TODO etudiant : cout minimal obtenu
int? varieteOpt = null;     // TODO etudiant : variete (nb de categories distinctes) au cout minimal
Console.WriteLine("Exercice 1 a completer : cout minimal vs variete maximale (front de Pareto).");
// --- Garde : verifie les valeurs declarees (inactive tant que l'exercice n'est pas complete).
if (coutOpt is double c1 && varieteOpt is int v1) {
    bool ok = c1 >= 0 && v1 >= 1 && v1 <= 5;
    Console.WriteLine(ok ? $"Garde ex.1 : OK (cout={c1}, variete={v1} coherents)."
                         : $"Garde ex.1 : ECHEC (cout={c1} doit etre >= 0, variete={v1} doit etre dans [1, 5]).");
} else Console.WriteLine("Garde ex.1 : inactive (a completer).");

Exercice 1 a completer : cout minimal vs variete maximale (front de Pareto).


Garde ex.1 : inactive (a completer).


### Exercice 2 — Préférence patient multi-niveaux (MaxSAT hiérarchique)

Le patient a des préférences **par ordre de priorité** : (1) aucun fruit à coque,
(2) préfère le poisson à la viande rouge, (3) aime le chocolat. Modélisez-les avec
`AssertSoft` à **poids croissants** (ex. 100 / 10 / 1) — le solveur satisfera d'abord les
priorités élevées. Comparez avec `Set("priority", "lex")` (domination lexicographique stricte).

In [43]:
// Exercice 2 -- preferences patient multi-niveaux (MaxSAT hierarchique).
// Indice : 3 AssertSoft a poids croissants (100/10/1) sur des clauses de preference,
//          puis comparer avec Set("priority", "lex") pour la domination lexicographique.
// Etape 1 : (poids 100) aucun fruit a coque. Etape 2 : (poids 10) poisson plutot que viande rouge.
// Etape 3 : (poids 1) le chocolat. Puis lire la penalite totale a l'optimum.
uint? penaliteTotale = null;   // TODO etudiant : somme des poids des preferences violees a l'optimum
Console.WriteLine("Exercice 2 a completer : MaxSAT hierarchique (poids croissants) vs priority=lex.");
// --- Garde : la penalite ne peut pas depasser la somme de tous les poids (100+10+1).
if (penaliteTotale is uint p2) {
    bool ok = p2 <= 111;
    Console.WriteLine(ok ? $"Garde ex.2 : OK (penalite totale {p2} <= penalite maximale 111)."
                         : $"Garde ex.2 : ECHEC (penalite totale {p2} > 111 : au moins une clause compte deux fois).");
} else Console.WriteLine("Garde ex.2 : inactive (a completer).");

Exercice 2 a completer : MaxSAT hierarchique (poids croissants) vs priority=lex.


Garde ex.2 : inactive (a completer).


### Exercice 3 — Optimiser le plan hebdomadaire en protéines (borne `box`)

Le §6 minimise le sel du plan semaine. Utilisez `Set("priority", "box")` pour obtenir, sur le
même encodage, le **maximum absolu de protéines** (sans égard au sel) ET le **minimum
absolu de sel** — les deux bornes qui cadrent le compromis hebdomadaire.

In [44]:
// Exercice 3 -- bornes box sur le plan hebdomadaire.
// Indice : reutiliser l'encodage selW du S6, Set("priority", "box"),
//          MkMinimize(weekSel) + MkMaximize(weekProt), un seul Check, lire h_sel.Upper et h_prot.Upper.
long? borneSelMin = null;     // TODO etudiant : sel hebdomadaire minimal absolu
long? borneProtMax = null;    // TODO etudiant : proteines hebdomadaires maximales absolues
Console.WriteLine("Exercice 3 a completer : bornes independantes (box) sel-min / prot-max du plan semaine.");
// --- Garde : deux bornes non negatives, et le sel-min ne peut pas depasser le sel d'un menu x 7.
if (borneSelMin is long s3 && borneProtMax is long p3) {
    bool ok = s3 >= 0 && p3 >= 0;
    Console.WriteLine(ok ? $"Garde ex.3 : OK (bornes sel-min={s3} g, prot-max={p3} g)."
                         : $"Garde ex.3 : ECHEC (bornes negatives : {s3} / {p3}).");
} else Console.WriteLine("Garde ex.3 : inactive (a completer).");

Exercice 3 a completer : bornes independantes (box) sel-min / prot-max du plan semaine.


Garde ex.3 : inactive (a completer).
